# K-Means Clustering

K-Means is an **unsupervised** algorithm: given *unlabeled* data, it groups the points into `k` clusters by similarity. **You** pick `k`; the algorithm discovers the cluster centers on its own — no target column, no `y`, is ever used to fit.

This is the one *unsupervised* notebook in this folder. The five algorithms before it (linear/logistic regression, decision tree, random forest, k-NN) all learned from **labels**. K-Means learns from the **geometry of the data alone**.

**Topics covered in this notebook**

1. Intuition — the assign ↔ update loop and the *inertia* objective
2. Clustering a dataset (recovering blobs with `make_blobs`)
3. Choosing `k` — the elbow method (a visual technique for finding a good number of clusters)
4. When to use it

## 1. Intuition

Imagine scattering `k` "magnets" (called **centroids**) onto a cloud of points. K-Means then repeats two steps until nothing moves — this is exactly an **Expectation–Maximization (EM)** style loop:

1. **Assign** (E-step): attach each point to its *nearest* centroid.
2. **Update** (M-step): move each centroid to the *mean* of the points now assigned to it.

Repeat. Assignments shift, centroids drift toward the dense parts of each group, and after a few passes the picture stabilizes.

### The objective it minimizes — *inertia*

Every iteration lowers the **inertia**, a.k.a. the **within-cluster sum of squares (WCSS)** — the total squared distance from each point to the centroid it was assigned to:

$$\text{inertia} \;=\; \sum_{i=1}^{n} \bigl\lVert x_i - \mu_{c(i)} \bigr\rVert^2
\;=\; \sum_{j=1}^{k} \sum_{x_i \in C_j} \bigl\lVert x_i - \mu_j \bigr\rVert^2$$

where $\mu_j$ is the centroid of cluster $C_j$ and $c(i)$ is the cluster assigned to point $x_i$. Lower inertia = tighter, more cohesive clusters.

### Two practical consequences

- Because it is **distance-based**, features should be on **comparable scales** (otherwise a large-range feature dominates the distance).
- The final result depends on the **random starting positions** of the centroids, so scikit-learn runs the whole thing several times and keeps the lowest-inertia solution. We set that number explicitly with **`n_init`** (this also avoids a version-dependent default-value warning).

## 2. Clustering a Dataset

We generate unlabeled 2-D data with three natural blobs and ask K-Means to recover three clusters. Crucially, **we never pass labels** — the model only ever sees `X`. That is what makes this *unsupervised*.

In [ ]:
# make_blobs: scikit-learn's synthetic-data generator. It draws isotropic (round) Gaussian
# "blobs" -- perfect, label-free test data for clustering (no download, fully offline).
from sklearn.datasets import make_blobs

# KMeans: the clustering estimator itself.
from sklearn.cluster import KMeans

In [ ]:
# Generate 300 points arranged in 3 natural groups.
#   centers=3      -> three underlying blobs
#   cluster_std    -> spread of each blob (smaller = tighter, better separated)
#   random_state   -> fixes the RNG so the dataset is identical on every run
#
# make_blobs returns (X, y_true). We deliberately throw the true labels away into `_`:
# clustering is UNSUPERVISED, so the ground-truth groups must NOT be fed to the model.
X, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# X is a plain feature matrix of shape (300, 2): 300 points, 2 coordinates each.
print("data shape:", X.shape)

In [ ]:
# Ask K-Means to find 3 clusters.
#   n_clusters=3   -> the k we choose up front (we picked 3 to match the blobs)
#   random_state   -> reproducible centroid initialization
#   n_init=10      -> run the whole assign/update loop 10 times from different random
#                     starts and keep the lowest-inertia result. Setting this EXPLICITLY
#                     avoids the version-dependent 'n_init' default-change warning.
km = KMeans(n_clusters=3, random_state=42, n_init=10)

# fit_predict = fit the centroids on X, then return each point's cluster index (0/1/2).
# Note we pass only X -- no labels anywhere.
labels = km.fit_predict(X)

# The labels are arbitrary group IDs (cluster "2" isn't better than "0"); they just say
# which points ended up together.
print("cluster label for first 10 points:", labels[:10])

In [ ]:
# cluster_centers_ holds the final centroid coordinates -- one (x, y) per cluster.
# These are the "magnets" after they stopped moving: each is the MEAN of its members.
print("cluster centers (one row per cluster):\n", km.cluster_centers_.round(2))

In [ ]:
# inertia_ is the value of the objective from section 1: the within-cluster sum of
# squared distances. It is what K-Means drove downward. Lower = tighter clusters.
print("inertia (within-cluster sum of squares):", round(km.inertia_, 1))

## 3. Choosing k (the Elbow Method)

K-Means needs you to pick `k` **before** it runs — but real data doesn't come with a label saying how many groups it has. The elbow method is a simple way to guess.

Inertia **always** drops as `k` grows (more centroids can always sit closer to more points — in the extreme, `k = n` gives inertia 0). So we don't look for the *lowest* inertia; we look for the **elbow**: the `k` after which adding another cluster stops buying a big improvement, i.e. where the steep drop flattens into a gentle slope.

Below, the sharp fall levels off right after `k = 3`, which matches the three blobs we created.

**Reminder — what inertia measures.** Inertia scores how tightly packed and cohesive the points are inside their clusters. It sums the squared distances from every point to its assigned centroid, so a smaller inertia means points sit closer to their group's center.

In [ ]:
# Fit K-Means for several candidate values of k and watch how inertia falls.
# The k where the decline suddenly slows is the "elbow" -- our estimate of the true count.
for k in range(1, 7):
    # Same explicit n_init here so every fit is warning-free and reproducible.
    m = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    print(f"k={k} -> inertia {m.inertia_:9.1f}")

In [ ]:
# Visualization: (left) the clusters K-Means found, (right) the elbow curve.
import matplotlib.pyplot as plt

# Recompute the inertia curve here so this plotting cell is fully self-contained
# (it does not depend on the print loop above having run).
ks = range(1, 7)
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_ for k in ks]

# This cell owns its OWN figure/axes -- robust plotting, no reliance on global state.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- left: clusters recovered by K-Means, colored by PREDICTED label -------------
# c=labels colors each point by the cluster K-Means assigned it (not the true group).
ax1.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=30, alpha=0.7)
# Overlay the final centroids as big red X's so we can see where the "magnets" settled.
ax1.scatter(
    km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
    c="red", marker="X", s=250, edgecolor="black", linewidth=1.5, label="centroids",
)
ax1.set_title("K-Means clusters (k=3)")
ax1.set_xlabel("feature 1")
ax1.set_ylabel("feature 2")
ax1.legend()

# --- right: elbow curve -- inertia vs. number of clusters ------------------------
ax2.plot(ks, inertias, marker="o", color="steelblue")
# Highlight k=3 (index 2), the elbow where the steep drop flattens out.
ax2.scatter([3], [inertias[2]], color="red", s=180, zorder=5, label="elbow (k=3)")
ax2.set_title("Elbow method")
ax2.set_xlabel("number of clusters (k)")
ax2.set_ylabel("inertia")
ax2.legend()

plt.tight_layout()
plt.show()

## 4. When to Use It

- **Unsupervised grouping when you have no labels** — customer segmentation, document/topic grouping, image color quantization, anomaly pre-screening.
- **Fast and scalable** — one of the cheapest clustering algorithms, comfortable on large datasets.
- **Requirements & caveats:**
  - You must **choose `k`** yourself (use the elbow method, or a silhouette score).
  - Features should be **scaled**, since it relies on Euclidean distance.
  - It assumes roughly **round, similarly-sized** clusters — it struggles with elongated, nested, or uneven-density shapes (reach for **DBSCAN** or **Gaussian mixtures** there).
- **No "accuracy."** Unlike the supervised algorithms earlier in this folder, there is no ground truth to score against at fit time — you judge quality with **inertia**, **silhouette score**, or plain domain sense.